In [1]:
import pandas as pd

from pathlib import Path


# Diretório principal do projeto
PROJECT_ROOT = Path.cwd().parent


# Origem: arquivos JSON da camada Bronze
SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "open_meteo"
    / "archive"
)


# Destino: arquivos Parquet da camada Silver
TARGET_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "open_meteo"
    / "archive"
)


# Conferir os caminhos
print("Diretório do projeto:", PROJECT_ROOT)
print("Origem Bronze:", SOURCE_DIR)
print("Destino Silver:", TARGET_DIR)

print("\nBronze existe?", SOURCE_DIR.exists())
print("Silver existe?", TARGET_DIR.exists())

Diretório do projeto: /home/mariarita-resende/Projetos/open_meteo_ingestion
Origem Bronze: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive
Destino Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive

Bronze existe? True
Silver existe? True


In [2]:
def get_archive_files(directory: Path) -> list[Path]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Diretório Bronze não encontrado: {directory}"
        )

    return sorted(directory.rglob("*.json"))


archive_files = get_archive_files(SOURCE_DIR)

print(f"Total de arquivos JSON encontrados: {len(archive_files)}")

print("\nExemplos de arquivos:")

for path in archive_files[:5]:
    print(path)

Total de arquivos JSON encontrados: 68

Exemplos de arquivos:
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive/20260423/amsterdam_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive/20260423/barcelona_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive/20260423/berlin_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive/20260423/brussels_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/archive/20260423/budapest_20260423.json


In [3]:
def build_archive_dataframe(path: Path) -> pd.DataFrame:
    path = Path(path)

    # Ler o arquivo JSON
    raw = pd.read_json(path)

    # Extrair os dados meteorológicos por hora
    hourly = raw.get("hourly", {})

    if isinstance(hourly, pd.Series):
        hourly = hourly.to_dict()

    if not isinstance(hourly, dict):
        return pd.DataFrame()

    # Obter os horários das observações
    times = pd.Series(hourly.get("time", [])).tolist()

    if not times:
        return pd.DataFrame()

    # Mapear os nomes dos campos originais para a Silver
    field_mapping = {
        "temperature_2m": "temperature_celsius",
        "apparent_temperature": "feels_like_celsius",
        "relative_humidity_2m": "relative_humidity_pct",
        "dew_point_2m": "dew_point_2m",
        "precipitation": "precipitation_mm",
        "rain": "rain",
        "snowfall": "snowfall",
        "snow_depth": "snow_depth",
        "weather_code": "weather_code",
        "wind_speed_10m": "wind_speed_10m",
        "wind_direction_10m": "wind_direction_10m",
    }

    # Criar as colunas de identificação
    data = {
        "source_name": "open_meteo",
        "source_type": "archive",
        "city_name": path.stem.rsplit("_", 1)[0],
        "ingestion_date": path.parent.name,
        "observation_time": [
            f"{observation_time}Z"
            for observation_time in times
        ],
    }

    # Adicionar as variáveis meteorológicas
    for source_field, target_field in field_mapping.items():
        values = hourly.get(source_field, [])

        if values is None:
            values = []

        values = list(values)

        # Garantir que todas as colunas tenham o mesmo número de linhas
        if len(values) < len(times):
            values += [None] * (len(times) - len(values))
        else:
            values = values[:len(times)]

        data[target_field] = values

    return pd.DataFrame(data)

In [4]:
sample_path = archive_files[0]

sample_df = build_archive_dataframe(sample_path)

print("Arquivo de origem:", sample_path.name)
print("Quantidade de linhas:", len(sample_df))
print("Quantidade de colunas:", len(sample_df.columns))

print("\nColunas:")
print(sample_df.columns.tolist())

print("\nPrimeiras cinco linhas:")
display(sample_df.head())

print("\nValores ausentes por coluna:")
print(sample_df.isna().sum())

Arquivo de origem: amsterdam_20260423.json
Quantidade de linhas: 168
Quantidade de colunas: 16

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'dew_point_2m', 'precipitation_mm', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Primeiras cinco linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,dew_point_2m,precipitation_mm,rain,snowfall,snow_depth,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,archive,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,5.5,0.0,0.0,0.0,0.0,0,10.8,20
1,open_meteo,archive,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,5.0,0.0,0.0,0.0,0.0,0,9.4,36
2,open_meteo,archive,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,5.3,0.0,0.0,0.0,0.0,0,11.2,33
3,open_meteo,archive,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,5.1,0.0,0.0,0.0,0.0,0,10.4,28
4,open_meteo,archive,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,5.1,0.0,0.0,0.0,0.0,0,9.4,14



Valores ausentes por coluna:
source_name              0
source_type              0
city_name                0
ingestion_date           0
observation_time         0
temperature_celsius      0
feels_like_celsius       0
relative_humidity_pct    0
dew_point_2m             0
precipitation_mm         0
rain                     0
snowfall                 0
snow_depth               0
weather_code             0
wind_speed_10m           0
wind_direction_10m       0
dtype: int64


In [5]:
print(sample_df.iloc[1].to_dict())

print("\nValores ausentes na segunda linha:")
print(sample_df.iloc[1].isna().sum())

print("\nTotal de valores ausentes no DataFrame:")
print(sample_df.isna().sum().sum())

{'source_name': 'open_meteo', 'source_type': 'archive', 'city_name': 'amsterdam', 'ingestion_date': '20260423', 'observation_time': '2026-04-23T01:00Z', 'temperature_celsius': 8.7, 'feels_like_celsius': 6.1, 'relative_humidity_pct': 78, 'dew_point_2m': 5.0, 'precipitation_mm': 0.0, 'rain': 0.0, 'snowfall': 0.0, 'snow_depth': 0.0, 'weather_code': 0, 'wind_speed_10m': 9.4, 'wind_direction_10m': 36}

Valores ausentes na segunda linha:
0

Total de valores ausentes no DataFrame:
0


In [6]:
total_files = 0
total_rows = 0
skipped = []

for path in archive_files:
    try:
        df = build_archive_dataframe(path)

        if df.empty:
            skipped.append(str(path))
            print(f"Arquivo vazio, ignorado: {path.name}")
            continue

        # Manter a organização por data
        ingestion_date = path.parent.name
        output_dir = TARGET_DIR / ingestion_date
        output_dir.mkdir(parents=True, exist_ok=True)

        # Manter o nome original, alterando a extensão
        output_path = output_dir / f"{path.stem}.parquet"

        # Gravar os dados em Parquet
        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False,
        )

        total_files += 1
        total_rows += len(df)

        print(f"Gravado: {output_path.name} | {len(df)} linhas")

    except Exception as error:
        print(f"Erro ao processar {path.name}: {error}")
        raise

print("\n--- Processamento concluído ---")
print(f"Arquivos gravados: {total_files}")
print(f"Total de linhas gravadas: {total_rows}")
print(f"Arquivos vazios ignorados: {len(skipped)}")

Gravado: amsterdam_20260423.parquet | 168 linhas
Gravado: barcelona_20260423.parquet | 168 linhas
Gravado: berlin_20260423.parquet | 168 linhas
Gravado: brussels_20260423.parquet | 168 linhas
Gravado: budapest_20260423.parquet | 168 linhas
Gravado: copenhagen_20260423.parquet | 168 linhas
Gravado: dublin_20260423.parquet | 168 linhas
Gravado: lisbon_20260423.parquet | 168 linhas
Gravado: paris_20260423.parquet | 168 linhas
Gravado: prague_20260423.parquet | 168 linhas
Gravado: rome_20260423.parquet | 168 linhas
Gravado: vienna_20260423.parquet | 168 linhas
Gravado: amsterdam_20260526.parquet | 24 linhas
Gravado: barcelona_20260526.parquet | 24 linhas
Gravado: berlin_20260526.parquet | 24 linhas
Gravado: brussels_20260526.parquet | 24 linhas
Gravado: budapest_20260526.parquet | 24 linhas
Gravado: copenhagen_20260526.parquet | 24 linhas
Gravado: dublin_20260526.parquet | 24 linhas
Gravado: lisbon_20260526.parquet | 24 linhas
Gravado: paris_20260526.parquet | 24 linhas
Gravado: prague_202

In [7]:
# Localizar todos os arquivos Parquet da Silver
parquet_files = sorted(TARGET_DIR.rglob("*.parquet"))

print("Total de arquivos Parquet:", len(parquet_files))

# Conferir se a quantidade corresponde à esperada
assert len(parquet_files) == total_files, (
    "A quantidade de arquivos encontrada é diferente "
    "da quantidade gravada."
)

# Ler um arquivo Parquet para validar sua integridade
sample_path = parquet_files[0]
sample_df = pd.read_parquet(sample_path)

print("\nArquivo testado:", sample_path.name)
print("Quantidade de linhas:", len(sample_df))
print("Quantidade de colunas:", len(sample_df.columns))

print("\nColunas:")
print(sample_df.columns.tolist())

print("\nPrimeiras cinco linhas:")
display(sample_df.head())

print("\nValores ausentes por coluna:")
print(sample_df.isna().sum())

print("\nValidação concluída!")

Total de arquivos Parquet: 68

Arquivo testado: amsterdam_20260423.parquet
Quantidade de linhas: 168
Quantidade de colunas: 16

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'dew_point_2m', 'precipitation_mm', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Primeiras cinco linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,dew_point_2m,precipitation_mm,rain,snowfall,snow_depth,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,archive,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,5.5,0.0,0.0,0.0,0.0,0,10.8,20
1,open_meteo,archive,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,5.0,0.0,0.0,0.0,0.0,0,9.4,36
2,open_meteo,archive,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,5.3,0.0,0.0,0.0,0.0,0,11.2,33
3,open_meteo,archive,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,5.1,0.0,0.0,0.0,0.0,0,10.4,28
4,open_meteo,archive,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,5.1,0.0,0.0,0.0,0.0,0,9.4,14



Valores ausentes por coluna:
source_name              0
source_type              0
city_name                0
ingestion_date           0
observation_time         0
temperature_celsius      0
feels_like_celsius       0
relative_humidity_pct    0
dew_point_2m             0
precipitation_mm         0
rain                     0
snowfall                 0
snow_depth               0
weather_code             0
wind_speed_10m           0
wind_direction_10m       0
dtype: int64

Validação concluída!
